# RQ3 — GeoReg-Curvature confirmatory run

This notebook runs the locked RQ3 protocol. Only the four anchors receive gradients. Seed 0 is used for gradient calibration and anchor-only lambda selection; seeds 1 and 2 are confirmatory. The 12 intermediate widths are not evaluated until lambda has been frozen.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2 for this protocol'
print('Detected GPUs:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')

## Secure clone

Attach the existing checkpoint dataset and create a Kaggle secret named `github_token`.

In [ ]:
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
assert (PROJECT_ROOT / 'scripts' / 'run_georeg_rq3.py').is_file(), 'Commit and push the RQ3 files first'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1', 'tabulate>=0.9'], check=True)

## Preflight and freeze the output location

In [ ]:
from datetime import datetime, timezone
import json, numpy as np, pandas as pd, yaml
from baseline_artifacts import find_confirmatory_root
INPUT_ROOT = Path('/kaggle/input/datasets/dyhngg/checkpoint-new-prune')
BASELINE_ROOT = find_confirmatory_root(INPUT_ROOT)
SOURCE_CONFIG = PROJECT_ROOT / 'configs' / 'kaggle_georeg_rq3.yaml'
config = yaml.safe_load(SOURCE_CONFIG.read_text())
assert config['experiment']['development_seed'] == 0
assert config['experiment']['confirmatory_seeds'] == [1, 2]
assert config['dataset']['num_workers'] == 0, 'Exact augmentation replay requires num_workers=0'
assert config['compression']['train_widths'] == [0.25, 0.50, 0.75, 1.00]
assert len(config['compression']['eval_widths']) == 16
assert config['georeg']['sanity_multipliers'] == [0.0, 0.5, 1.0, 2.0]
assert config['georeg']['sanity_epoch'] == 10 and config['training']['epochs'] == 20
for seed in [0, 1, 2]:
    assert (BASELINE_ROOT / f'seed_{seed}' / 'checkpoint.pt').is_file()
    for tag in ['025', '050', '075', '100']:
        assert (BASELINE_ROOT / f'seed_{seed}' / 'features' / f'features_budget_{tag}.pt').is_file()
RUN_NAME = datetime.now(timezone.utc).strftime('kaggle-georeg-rq3-%Y%m%d-%H%M%S')
RUN_DIR = Path('/kaggle/working/new-pruning-outputs') / RUN_NAME
config['experiment']['output_dir'] = str(RUN_DIR)
RESOLVED_CONFIG = Path('/kaggle/working/kaggle_georeg_rq3_resolved.yaml')
RESOLVED_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print('Baseline:', BASELINE_ROOT)
print('Run directory:', RUN_DIR)
print('Protocol preflight: OK')

## Execute calibration → sanity gate → final run

If no nonzero lambda passes the preregistered anchor-only gates, execution stops before opening the dense grid and produces a SANITY NO-GO report. Completed stages are reused when this cell is rerun with the same `RUN_DIR`.

In [ ]:
import importlib, georeg_curvature
import scripts.run_georeg_rq3 as rq3_runner
importlib.reload(georeg_curvature)
rq3_runner = importlib.reload(rq3_runner)
started = time.perf_counter()
result = rq3_runner.run_rq3(RESOLVED_CONFIG, gpu_ids=[0, 1])
print(f"RQ3 workflow completed in {(time.perf_counter() - started) / 3600:.2f} hours")
print(result['decision'])

## Inspect the preregistered decision

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RUN_DIR / 'rq3_report.md').read_text()))
display(result['sanity'])
if result['decision'].get('sanity_go'):
    display(pd.read_csv(RUN_DIR / 'rq3_main_table.csv'))
    display(pd.read_csv(RUN_DIR / 'rq3_paired_comparison.csv'))
    display(pd.read_csv(RUN_DIR / 'rq3_confirmatory_seed_checks.csv'))

In [ ]:
import matplotlib.pyplot as plt
if result['decision'].get('sanity_go'):
    baseline = pd.read_csv(BASELINE_ROOT / 'central_analysis_all_seeds.csv')
    georeg = pd.read_csv(RUN_DIR / 'georeg_central_analysis_all_seeds.csv')
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for label, frame in [('Baseline', baseline), ('GeoReg-Curvature', georeg)]:
        accuracy = frame.groupby('budget')['accuracy'].mean()
        geometry = frame.dropna(subset=['local_wasserstein_sensitivity']).groupby('budget')['local_wasserstein_sensitivity'].mean()
        axes[0].plot(accuracy.index, accuracy, marker='o', label=label)
        axes[1].plot(geometry.index, geometry, marker='o', label=label)
    axes[0].set(xlabel='Width', ylabel='Mean test accuracy', title='Dense-budget accuracy')
    axes[1].set(xlabel='Interval start c', ylabel='Mean G(c)', title='Local Wasserstein geometry')
    for axis in axes:
        axis.grid(alpha=.25); axis.legend()
    fig.tight_layout()
    fig.savefig(RUN_DIR / 'rq3_summary.png', dpi=180)
    plt.show()

## Package all protocol, checkpoints and results

In [ ]:
import shutil, zipfile
from IPython.display import FileLink
files = sorted(path for path in RUN_DIR.rglob('*') if path.is_file())
pd.DataFrame({'relative_path': [str(path.relative_to(RUN_DIR)) for path in files], 'size_bytes': [path.stat().st_size for path in files]}).to_csv(RUN_DIR / 'artifact_manifest.csv', index=False)
archive = Path(shutil.make_archive(str(Path('/kaggle/working') / RUN_NAME), 'zip', root_dir=RUN_DIR))
with zipfile.ZipFile(archive) as bundle:
    names = set(bundle.namelist())
required = ['resolved_config.yaml', 'protocol/train_projection_directions.pt', 'protocol/heldout_projection_directions.pt', 'development/sanity_selection.csv', 'development/lambda_selection.json', 'rq3_decision.json', 'rq3_report.md']
if result['decision'].get('sanity_go'):
    required += ['rq3_main_table.csv', 'rq3_paired_comparison.csv', 'rq3_confirmatory_seed_checks.csv', 'georeg/seed_0/checkpoint.pt', 'georeg/seed_1/checkpoint.pt', 'georeg/seed_2/checkpoint.pt']
missing = [name for name in required if name not in names]
assert not missing, f'Missing required artifacts: {missing}'
print('Archive:', archive)
display(FileLink(str(archive)))
print('Use Save Version after this cell completes.')